In [ ]:
# fine-tuned 모델 체크포인트 확인
ls /project_antwerp/hbae/Loki_output/0228_loki_finetune_10fold/fold_01/

# zero-shot embedding 폴더 확인
ls /project_antwerp/hbae/Loki_output/0228_embeddings_zeroshot/fold_01/ | head -5

# ST training embeddings 확인
ls /project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_01/ | head -10

# ref_file.csv 행 수 (슬라이드 몇 개?)
wc -l /project_antwerp/hbae/ref_file.csv

# TCGA patch 폴더 슬라이드 수
ls /project_antwerp/TCGA-HNSC/TCGA_patch/ | wc -l
ls: cannot access '/project_antwerp/hbae/Loki_output/0228_loki_finetune_10fold/fold_01/': No such file or directory
per_sample_dist
train_exprs.npy
train_img_embs.npy
train_text_embs.npy
val_exprs.npy
fold01_violin_plot.png
sample_gene_pcc_dist.npy
train_exprs.npy
train_img_embs.npy
train_text_embs.npy
val_exprs.npy
val_img_embs.npy
462 /project_antwerp/hbae/ref_file.csv
528

In [ ]:
# patch 폴더 이름 몇 개 확인
ls /project_antwerp/TCGA-HNSC/TCGA_patch/ | head -5

# ref_file의 wsi_file_name에서 앞부분만 추출해서 비교
head -5 /project_antwerp/hbae/ref_file.csv | cut -d',' -f2 | cut -d'.' -f1
TCGA-4P-AA8J-01Z-00-DX1
TCGA-BA-4074-01Z-00-DX1
TCGA-BA-4074-01Z-00-DX1.377dc7cb-f029-4bf5-bdd5-8997f91e551a
TCGA-BA-4077-01Z-00-DX1
TCGA-BA-4077-01Z-00-DX1.604893b7-5fcb-4bab-b2cd-6cad1c8ab456
wsi_file_name
TCGA-CV-6950-01Z-00-DX1
TCGA-D6-6515-01Z-00-DX1
TCGA-D6-6515-01Z-00-DX2
TCGA-F7-7848-01Z-00-DX1

In [ ]:
# HDF5 구조 확인

import h5py
f = h5py.File('/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-4P-AA8J-01Z-00-DX1/TCGA-4P-AA8J-01Z-00-DX1.hdf5', 'r')
print('Keys:', list(f.keys()))
for k in f.keys():
    print(f'{k}: shape={f[k].shape}, dtype={f[k].dtype}')
f.close()

In [2]:

import numpy as np
d = '/project_antwerp/hbae/Loki_output/0228_embeddings_zeroshot/fold_01/'
img = np.load(d+'train_img_embs.npy')
expr = np.load(d+'train_exprs.npy')
print('train_img_embs:', img.shape)
print('train_exprs:', expr.shape)


train_img_embs: (64157, 768)
train_exprs: (64157, 2000)


In [1]:

import torch
ckpt = torch.load(
    '/project_antwerp/hbae/Loki_output/0317_10epoch_finetune_10fold_runs_hvg_/fold_01/finetune_hvg_fold_01_20260320_212457/checkpoints/epoch_latest.pt',
    map_location='cpu', weights_only=False
)
print('Keys:', list(ckpt.keys()) if isinstance(ckpt, dict) else type(ckpt))
if isinstance(ckpt, dict):
    for k in ckpt.keys():
        print(f'  {k}: {type(ckpt[k])}')


Keys: ['epoch', 'name', 'state_dict', 'optimizer', 'scaler']
  epoch: <class 'int'>
  name: <class 'str'>
  state_dict: <class 'collections.OrderedDict'>
  optimizer: <class 'dict'>
  scaler: <class 'dict'>


In [2]:

import torch
ckpt = torch.load(
    '/project_antwerp/hbae/Loki_output/0317_10epoch_finetune_10fold_runs_hvg_/fold_01/finetune_hvg_fold_01_20260320_212457/checkpoints/epoch_latest.pt',
    map_location='cpu', weights_only=False
)
sd = ckpt['state_dict']
keys = list(sd.keys())
print('Total keys:', len(keys))
print('First 5 keys:', keys[:5])
print('Last 5 keys:', keys[-5:])
print('Epoch:', ckpt['epoch'])
print('Name:', ckpt['name'])


Total keys: 773
First 5 keys: ['logit_scale', 'text.cls_emb', 'text.positional_embedding', 'text.text_projection', 'text.token_embedding.weight']
Last 5 keys: ['text_decoder.cross_attn.11.mlp.c_fc.bias', 'text_decoder.cross_attn.11.mlp.c_proj.weight', 'text_decoder.cross_attn.11.mlp.c_proj.bias', 'text_decoder.ln_final.weight', 'text_decoder.ln_final.bias']
Epoch: 10
Name: finetune_hvg_fold_01_20260320_212457


In [3]:

import torch
import open_clip

# 모델 구조의 key 확인
model, _, _ = open_clip.create_model_and_transforms('coca_ViT-L-14')
model_keys = list(model.state_dict().keys())
print('Model first 5 keys:', model_keys[:5])
print('Model last 5 keys:', model_keys[-5:])
print('Total model keys:', len(model_keys))

# checkpoint key와 비교
ckpt = torch.load(
    '/project_antwerp/hbae/Loki_output/0317_10epoch_finetune_10fold_runs_hvg_/fold_01/finetune_hvg_fold_01_20260320_212457/checkpoints/epoch_latest.pt',
    map_location='cpu', weights_only=False
)
sd = ckpt['state_dict']
ckpt_keys = list(sd.keys())

# 겹치는 key 수 확인
overlap = set(model_keys) & set(ckpt_keys)
print('Overlapping keys:', len(overlap))
print('Model only:', len(set(model_keys) - set(ckpt_keys)))
print('Checkpoint only:', len(set(ckpt_keys) - set(model_keys)))


Model first 5 keys: ['logit_scale', 'text.cls_emb', 'text.positional_embedding', 'text.text_projection', 'text.token_embedding.weight']
Model last 5 keys: ['text_decoder.cross_attn.11.mlp.c_fc.bias', 'text_decoder.cross_attn.11.mlp.c_proj.weight', 'text_decoder.cross_attn.11.mlp.c_proj.bias', 'text_decoder.ln_final.weight', 'text_decoder.ln_final.bias']
Total model keys: 773
Overlapping keys: 773
Model only: 0
Checkpoint only: 0


In [4]:

import torch
import open_clip
import torch.nn.functional as F

device = 'cpu'

# 모델 생성 (WARNING은 무시)
model, _, preprocess = open_clip.create_model_and_transforms('coca_ViT-L-14')

# checkpoint 로드
ckpt = torch.load(
    '/project_antwerp/hbae/Loki_output/0317_10epoch_finetune_10fold_runs_hvg_/fold_01/finetune_hvg_fold_01_20260320_212457/checkpoints/epoch_latest.pt',
    map_location=device, weights_only=False
)
result = model.load_state_dict(ckpt['state_dict'], strict=True)
print('Missing keys:', result.missing_keys)
print('Unexpected keys:', result.unexpected_keys)

# logit_scale 확인 (fine-tuning 되면 값이 바뀜)
print('logit_scale:', model.logit_scale.item())

# zero-shot checkpoint와 비교
ckpt_zs = torch.load(
    '/project_antwerp/assets/loki_ckpts/checkpoint.pt',
    map_location=device, weights_only=False
)
sd_zs = ckpt_zs.get('state_dict', ckpt_zs.get('model', ckpt_zs))
print('Zero-shot logit_scale:', sd_zs.get('logit_scale', 'key not found'))

# 체크포인트는 로딩 잘됨


Missing keys: []
Unexpected keys: []
logit_scale: 4.472879409790039
Zero-shot logit_scale: tensor(4.4682)


In [3]:

import numpy as np
zs = np.load('/project_antwerp/hbae/Loki_output/0228_embeddings_zeroshot/fold_01/train_img_embs.npy')
ft = np.load('/project_antwerp/hbae/Loki_output/0317_epoch10_finetune_embedding_new/fold_01/train_img_embs.npy')
print('Zero-shot  mean:', zs.mean(), 'std:', zs.std())
print('Fine-tuned mean:', ft.mean(), 'std:', ft.std())
# print('Are they equal?', np.allclose(zs, ft))
print('Max diff:', np.abs(zs - ft).max())
#  fine-tuned train embeddings는 제대로 된 
# Zero-shot  mean: 0.0007251755 std: 0.036077138
# 0228Fine-tuned mean: 0.00034938485 std: 0.03608272
# Are they equal? False
# Max diff: 0.39707923

Zero-shot  mean: 0.0007251755 std: 0.036077138
Fine-tuned mean: 0.00032907058 std: 0.03608292


ValueError: operands could not be broadcast together with shapes (64157,768) (49156,768) 

In [10]:

import torch, numpy as np, h5py, glob
import torch.nn.functional as F
import open_clip
from PIL import Image

device = 'cuda'

# Zero-shot 모델 로드 (직접 로드)
model_zs, _, preprocess = open_clip.create_model_and_transforms('coca_ViT-L-14')
ckpt_zs = torch.load('/project_antwerp/assets/loki_ckpts/checkpoint.pt', map_location=device, weights_only=False)
sd_zs = ckpt_zs.get('state_dict', ckpt_zs.get('model', ckpt_zs))
model_zs.load_state_dict(sd_zs, strict=False)
model_zs = model_zs.to(device).eval()

# Fine-tuned 모델 로드
model_ft, _, _ = open_clip.create_model_and_transforms('coca_ViT-L-14')
ckpt_ft = torch.load('/project_antwerp/hbae/Loki_output/0317_10epoch_finetune_10fold_runs_hvg_/fold_01/finetune_hvg_fold_01_20260320_212457/checkpoints/epoch_latest.pt', map_location=device, weights_only=False)
model_ft.load_state_dict(ckpt_ft['state_dict'], strict=True)
model_ft = model_ft.to(device).eval()

# tile 5개 로드
hdf5 = glob.glob('/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-4P-AA8J-01Z-00-DX1/*.hdf5')[0]
with h5py.File(hdf5) as f:
    keys = list(f.keys())[:5]
    imgs = [Image.fromarray(f[k][:]) for k in keys]

tensors = torch.stack([preprocess(img) for img in imgs]).to(device)

with torch.no_grad():
    emb_zs = F.normalize(model_zs.encode_image(tensors), dim=-1)
    emb_ft = F.normalize(model_ft.encode_image(tensors), dim=-1)

print('ZS emb mean:', emb_zs.cpu().numpy().mean())
print('FT emb mean:', emb_ft.cpu().numpy().mean())
print('Are equal?', torch.allclose(emb_zs, emb_ft, atol=1e-4))
print('Max diff:', (emb_zs - emb_ft).abs().max().item())


ZS emb mean: 0.0018671175
FT emb mean: 0.0014122652
Are equal? False
Max diff: 0.1861700862646103


In [11]:
# val_exprs.npy로 top 300 확인 가능한지

import numpy as np

val_exprs = np.load('/project_antwerp/hbae/Loki_output/0228_embeddings_zeroshot/fold_01/val_exprs.npy')
print('val_exprs shape:', val_exprs.shape)


val_exprs shape: (7215, 2000)


In [2]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [4]:

import numpy as np
pred = np.load('/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/zeroshot_top300/slide_preds_top300.npy')
print('shape:', pred.shape)


shape: (331, 290)


In [5]:

import numpy as np
pred = np.load('/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/zeroshot/slide_preds_sliding_window.npy')
bulk = np.load('/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/zeroshot/slide_bulks.npy')
genes = np.load('/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/zeroshot/common_genes.npy', allow_pickle=True)
print('pred shape:', pred.shape)
print('bulk shape:', bulk.shape)
print('genes:', len(genes))


pred shape: (331, 1968)
bulk shape: (331, 1968)
genes: 1968


In [1]:

import pandas as pd, numpy as np
df = pd.read_csv('/project_antwerp/hbae/ref_file.csv', index_col=0)
rna = df[[c for c in df.columns if c.startswith('rna_')]].values
print('min:', rna.min(), 'max:', rna.max(), 'mean:', rna.mean())
print('first row sample:', rna[0, :5])


min: 0.0 max: 11.41565638847998 mean: 1.1278844013342597
first row sample: [2.2622005  3.25557798 0.46561903 0.76700155 0.96210504]


In [2]:

import pandas as pd
import numpy as np
import scanpy as sc

ref_df = pd.read_csv('/project_antwerp/hbae/ref_file.csv', index_col=0)
rna_cols = [c for c in ref_df.columns if c.startswith('rna_')]

# AnnData 생성
import anndata
adata = anndata.AnnData(X=ref_df[rna_cols].values.astype(float))
adata.var_names = [c.replace('rna_', '') for c in rna_cols]
adata.obs_names = ref_df.index.astype(str)

print('Shape:', adata.shape)
print('X range:', adata.X.min(), '~', adata.X.max())

# HVG 선택 (log된 데이터용 flavor='seurat')
sc.pp.highly_variable_genes(adata, n_top_genes=300, flavor='seurat')
hvg = adata.var_names[adata.var['highly_variable']].tolist()
print(f'HVG selected: {len(hvg)}')
print('Top 10 HVG:', hvg[:10])

# 저장
np.save('/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/tcga_bulk_hvg300.npy', hvg)
print('Saved!')


Shape: (461, 24778)
X range: 0.0 ~ 11.41565638847998
HVG selected: 300
Top 10 HVG: ['HSPB6', 'PDK4', 'SOX8', 'CCL26', 'NOS2', 'TKTL1', 'FMO3', 'LTF', 'FHL1', 'BIRC3']
Saved!


In [3]:

import numpy as np
import pandas as pd
from scipy.stats import pearsonr

# 저장된 예측값 로드
pred_arr     = np.load('/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/zeroshot/slide_preds_sliding_window.npy')
bulk_arr     = np.load('/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/zeroshot/slide_bulks.npy')
common_genes = np.load('/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/zeroshot/common_genes.npy', allow_pickle=True).tolist()

# TCGA HVG 300 로드
hvg300 = np.load('/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/tcga_bulk_hvg300.npy', allow_pickle=True).tolist()

# common_genes에서 HVG300 index
idx_B2 = [i for i, g in enumerate(common_genes) if g in set(hvg300)]
print(f'HVG300 in common_genes: {len(idx_B2)}')

p_sub = pred_arr[:, idx_B2]
b_sub = bulk_arr[:, idx_B2]

# Gene-wise PCC
gene_pccs = []
for i in range(p_sub.shape[1]):
    p, b = p_sub[:, i], b_sub[:, i]
    if p.std() < 1e-8 or b.std() < 1e-8:
        continue
    r, _ = pearsonr(p, b)
    gene_pccs.append(r)
gene_pccs = np.array(gene_pccs)

# Slide-wise PCC
slide_pccs = []
for i in range(p_sub.shape[0]):
    p, b = p_sub[i], b_sub[i]
    if p.std() < 1e-8 or b.std() < 1e-8:
        slide_pccs.append(np.nan)
        continue
    r, _ = pearsonr(p, b)
    slide_pccs.append(r)
slide_pccs = np.array(slide_pccs)
valid = ~np.isnan(slide_pccs)

print(f'[Method B2: TCGA bulk HVG300 (Scanpy Seurat)]')
print(f'  Gene-wise  | mean={gene_pccs.mean():.4f}  median={np.median(gene_pccs):.4f}  PCC>0.1: {(gene_pccs>0.1).sum()}  PCC>0.3: {(gene_pccs>0.3).sum()}')
print(f'  Slide-wise | mean={slide_pccs[valid].mean():.4f}  median={np.median(slide_pccs[valid]):.4f}')


HVG300 in common_genes: 87
[Method B2: TCGA bulk HVG300 (Scanpy Seurat)]
  Gene-wise  | mean=0.0350  median=0.0452  PCC>0.1: 16  PCC>0.3: 0
  Slide-wise | mean=0.2530  median=0.2271


In [4]:

import pandas as pd
import numpy as np
import anndata
import scanpy as sc

ref_df = pd.read_csv('/project_antwerp/hbae/ref_file.csv', index_col=0)

# HVG 2000개 gene list 로드
with open('/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt') as f:
    gene_list = [l.strip() for l in f if l.strip()]

# bulk에서 HVG 2000개와 겹치는 gene만 사용
rna_cols     = [c for c in ref_df.columns if c.startswith('rna_')]
ref_genes    = [c.replace('rna_', '') for c in rna_cols]
common_genes = [g for g in gene_list if g in ref_genes]
common_cols  = ['rna_' + g for g in common_genes]

print(f'Common genes: {len(common_genes)}')

# AnnData 생성 (HVG 2000 교집합만)
adata = anndata.AnnData(X=ref_df[common_cols].values.astype(float))
adata.var_names = common_genes
adata.obs_names = ref_df.index.astype(str)

# HVG 300 선택 (HVG 2000 안에서만)
sc.pp.highly_variable_genes(adata, n_top_genes=300, flavor='seurat')
hvg300 = adata.var_names[adata.var['highly_variable']].tolist()
print(f'HVG selected: {len(hvg300)}')
print('Top 10:', hvg300[:10])

np.save('/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/tcga_bulk_hvg300_from_common.npy', hvg300)
print('Saved!')


Common genes: 1968
HVG selected: 301
Top 10: ['ABCA3', 'ACKR4', 'ACTC1', 'ACTG2', 'ACTN2', 'ADGRF1', 'ADRA2A', 'AGR2', 'AKAP12', 'AKR1C3']
Saved!


In [5]:

import numpy as np
import pandas as pd
from scipy.stats import pearsonr

pred_arr     = np.load('/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/zeroshot/slide_preds_sliding_window.npy')
bulk_arr     = np.load('/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/zeroshot/slide_bulks.npy')
common_genes = np.load('/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/zeroshot/common_genes.npy', allow_pickle=True).tolist()

hvg300 = np.load('/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/tcga_bulk_hvg300_from_common.npy', allow_pickle=True).tolist()

idx_B = [i for i, g in enumerate(common_genes) if g in set(hvg300)]
print(f'HVG300 in common_genes: {len(idx_B)}')

p_sub = pred_arr[:, idx_B]
b_sub = bulk_arr[:, idx_B]

# Gene-wise PCC
gene_pccs = []
for i in range(p_sub.shape[1]):
    p, b = p_sub[:, i], b_sub[:, i]
    if p.std() < 1e-8 or b.std() < 1e-8:
        continue
    r, _ = pearsonr(p, b)
    gene_pccs.append(r)
gene_pccs = np.array(gene_pccs)

# Slide-wise PCC
slide_pccs = []
for i in range(p_sub.shape[0]):
    p, b = p_sub[i], b_sub[i]
    if p.std() < 1e-8 or b.std() < 1e-8:
        slide_pccs.append(np.nan)
        continue
    r, _ = pearsonr(p, b)
    slide_pccs.append(r)
slide_pccs = np.array(slide_pccs)
valid = ~np.isnan(slide_pccs)

print(f'[Method B: TCGA HVG300 from common 1968]')
print(f'  Gene-wise  | mean={gene_pccs.mean():.4f}  median={np.median(gene_pccs):.4f}  PCC>0.1: {(gene_pccs>0.1).sum()}  PCC>0.3: {(gene_pccs>0.3).sum()}')
print(f'  Slide-wise | mean={slide_pccs[valid].mean():.4f}  median={np.median(slide_pccs[valid]):.4f}')


HVG300 in common_genes: 301
[Method B: TCGA HVG300 from common 1968]
  Gene-wise  | mean=0.0206  median=0.0317  PCC>0.1: 47  PCC>0.3: 0
  Slide-wise | mean=0.6023  median=0.6236


In [1]:
# train_exprs 스케일 확인

import numpy as np
e = np.load('/project_antwerp/hbae/Loki_output/0228_embeddings_zeroshot/fold_01/train_exprs.npy')
print('min:', e.min(), 'max:', e.max(), 'mean:', e.mean())


min: 0.0 max: 8.288818 mean: 0.29060665


In [2]:

import numpy as np

pred = np.load('/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/zeroshot/slide_preds_sliding_window.npy')
bulk = np.load('/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/zeroshot/slide_bulks.npy')

print('=== Prediction ===')
print('min:', pred.min(), 'max:', pred.max(), 'mean:', pred.mean())
print('std:', pred.std())

print('=== Bulk ===')
print('min:', bulk.min(), 'max:', bulk.max(), 'mean:', bulk.mean())
print('std:', bulk.std())

print('=== Variance ratio ===')
print('pred var / bulk var:', pred.var() / bulk.var())


=== Prediction ===
min: 0.012847692 max: 3.8935783 mean: 0.28532684
std: 0.37149036
=== Bulk ===
min: 0.0 max: 11.41565638847998 mean: 1.8567000619453717
std: 1.5219638292279813
=== Variance ratio ===
pred var / bulk var: 0.059578072819028445


In [1]:

import numpy as np
import pandas as pd
from scipy.stats import pearsonr

# fine-tuned fold_01 결과 로드
pred = np.load('/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/finetuned_ensemble/slide_preds_ensemble.npy')
bulk = np.load('/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/finetuned_ensemble/slide_bulks.npy')
genes = np.load('/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/finetuned_ensemble/common_genes.npy', allow_pickle=True).tolist()

print('pred shape:', pred.shape)
print('bulk shape:', bulk.shape)
print('genes:', len(genes))


pred shape: (331, 1968)
bulk shape: (331, 1968)
genes: 1968


In [2]:

import numpy as np
import pandas as pd
from scipy.stats import pearsonr

GENE_LIST    = '/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt'
ZEROSHOT_EMB = '/project_antwerp/hbae/Loki_output/0228_embeddings_zeroshot/fold_01'
FT_DIR       = '/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/finetuned_ensemble'
HVG_PATH     = '/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/tcga_bulk_hvg300_from_common.npy'

# 로드
with open(GENE_LIST) as f:
    gene_list = [l.strip() for l in f if l.strip()]

pred  = np.load(FT_DIR + '/slide_preds_ensemble.npy')  # (331, 1968)
bulk  = np.load(FT_DIR + '/slide_bulks.npy')
genes = np.load(FT_DIR + '/common_genes.npy', allow_pickle=True).tolist()

def calc_pcc(pred, bulk, idx, label):
    p = pred[:, idx]
    b = bulk[:, idx]
    gpccs = []
    for i in range(p.shape[1]):
        if p[:,i].std() < 1e-8 or b[:,i].std() < 1e-8: continue
        r, _ = pearsonr(p[:,i], b[:,i])
        gpccs.append(r)
    spccs = []
    for i in range(p.shape[0]):
        if p[i].std() < 1e-8 or b[i].std() < 1e-8:
            spccs.append(np.nan); continue
        r, _ = pearsonr(p[i], b[i])
        spccs.append(r)
    g = np.array(gpccs)
    s = np.array(spccs)
    v = ~np.isnan(s)
    print(f'[{label}]')
    print(f'  Gene-wise  | mean={g.mean():.4f}  median={np.median(g):.4f}  PCC>0.1: {(g>0.1).sum()}  PCC>0.3: {(g>0.3).sum()}')
    print(f'  Slide-wise | mean={s[v].mean():.4f}  median={np.median(s[v]):.4f}')

# 방법 A: ST val_exprs top 300
val_exprs = np.load(ZEROSHOT_EMB + '/val_exprs.npy')
top300_A  = [gene_list[i] for i in np.argsort(val_exprs.mean(axis=0))[::-1][:300]]
idx_A     = [i for i, g in enumerate(genes) if g in set(top300_A)]
calc_pcc(pred, bulk, idx_A, 'Fine-tuned fold_01 | Method A: ST val_exprs top 300')

# 방법 B: TCGA HVG top 300
hvg300 = np.load(HVG_PATH, allow_pickle=True).tolist()
idx_B  = [i for i, g in enumerate(genes) if g in set(hvg300)]
calc_pcc(pred, bulk, idx_B, 'Fine-tuned fold_01 | Method B: TCGA HVG top 300')

# 방법 C: PCC top 300 (상한선)
all_pccs = []
for i in range(len(genes)):
    p, b = pred[:,i], bulk[:,i]
    if p.std() < 1e-8 or b.std() < 1e-8:
        all_pccs.append(-999); continue
    r, _ = pearsonr(p, b)
    all_pccs.append(r)
all_pccs = np.array(all_pccs)
valid_idx = np.where(all_pccs > -999)[0]
idx_C = valid_idx[np.argsort(all_pccs[valid_idx])[::-1][:300]].tolist()
calc_pcc(pred, bulk, idx_C, 'Fine-tuned fold_01 | Method C: PCC top 300 (post-hoc)')


[Fine-tuned fold_01 | Method A: ST val_exprs top 300]
  Gene-wise  | mean=0.0468  median=0.0383  PCC>0.1: 59  PCC>0.3: 0
  Slide-wise | mean=0.4950  median=0.5029
[Fine-tuned fold_01 | Method B: TCGA HVG top 300]
  Gene-wise  | mean=0.0363  median=0.0377  PCC>0.1: 74  PCC>0.3: 0
  Slide-wise | mean=0.6029  median=0.6251
[Fine-tuned fold_01 | Method C: PCC top 300 (post-hoc)]
  Gene-wise  | mean=0.1512  median=0.1440  PCC>0.1: 300  PCC>0.3: 0
  Slide-wise | mean=0.7447  median=0.7647


In [3]:

import numpy as np
ids = np.load('/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/finetuned_ensemble/slide_ids.npy', allow_pickle=True)
print('slides:', len(ids))
# folds used per slide 확인할 방법 없음 → 저장 안 됨


slides: 331


In [4]:

import numpy as np
e = np.load('/project_antwerp/hbae/Loki_output/0228_embeddings_zeroshot/fold_01/train_exprs.npy')
print('ST train_exprs - min:', e.min(), 'max:', e.max(), 'mean:', e.mean())

# bulk 비교
import pandas as pd
ref = pd.read_csv('/project_antwerp/hbae/ref_file.csv', index_col=0)
rna = ref[[c for c in ref.columns if c.startswith('rna_')]].values
print('Bulk ref_file  - min:', rna.min(), 'max:', rna.max(), 'mean:', rna.mean())


ST train_exprs - min: 0.0 max: 8.288818 mean: 0.29060665
Bulk ref_file  - min: 0.0 max: 11.41565638847998 mean: 1.1278844013342597


In [5]:

import h5py, numpy as np
f = h5py.File('/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-4P-AA8J-01Z-00-DX1/TCGA-4P-AA8J-01Z-00-DX1.hdf5', 'r')
keys = list(f.keys())[:20]
x_coords = sorted(set(int(k.split('_')[1]) for k in keys))
y_coords = sorted(set(int(k.split('_')[0]) for k in keys))
print('x 간격:', [x_coords[i+1]-x_coords[i] for i in range(min(5, len(x_coords)-1))])
print('y 간격:', [y_coords[i+1]-y_coords[i] for i in range(min(5, len(y_coords)-1))])
f.close()


x 간격: [512, 512, 512, 512, 512]
y 간격: []


In [6]:

import pandas as pd
from pathlib import Path
ref_df = pd.read_csv('/project_antwerp/hbae/ref_file.csv', index_col=0)
ref_df['slide_id'] = ref_df['wsi_file_name'].apply(lambda x: x.split('.')[0])
patch_dirs = {p.name for p in Path('/project_antwerp/TCGA-HNSC/TCGA_patch').iterdir()}
unmatched = ref_df[~ref_df['slide_id'].isin(patch_dirs)]
print(f'Unmatched: {len(unmatched)} / {len(ref_df)}')
print(unmatched['slide_id'].head(10).tolist())


Unmatched: 130 / 461
['TCGA-F7-7848-01Z-00-DX1', 'TCGA-BA-4078-01Z-00-DX1', 'TCGA-CN-4730-01Z-00-DX1', 'TCGA-D6-6823-01Z-00-DX1', 'TCGA-F7-8298-01Z-00-DX1', 'TCGA-HD-A4C1-01Z-00-DX1', 'TCGA-BA-5149-01Z-00-DX1', 'TCGA-CN-5364-01Z-00-DX1', 'TCGA-P3-A6SX-01Z-00-DX1', 'TCGA-BB-7861-01Z-00-DX1']


In [7]:

import scanpy as sc, os
files = [f for f in os.listdir('/project_antwerp/hbae/data/0317_HVG_NEW/') if f.endswith('.h5ad')]
print('h5ad files:', files[:3])
if files:
    adata = sc.read_h5ad('/project_antwerp/hbae/data/0317_HVG_NEW/' + files[0])
    print('uns keys:', list(adata.uns.keys()))
    print('X min/max/mean:', adata.X.min(), adata.X.max(), adata.X.mean())


h5ad files: []


In [2]:
import numpy as np, os
d = '/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/TCGA_embeddings/fold_01/'
files = [f for f in os.listdir(d) if not f.endswith('_coords.npy')][:3]
for f in files:
    e = np.load(d+f)
    print(f'{f}: {e.shape}')


TCGA-WA-A7GZ-01Z-00-DX2.npy: (2164, 768)
TCGA-CV-5978-01Z-00-DX1.npy: (7575, 768)
TCGA-MZ-A6I9-01Z-00-DX1.npy: (385, 768)


In [1]:

import numpy as np
pred = np.load('/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/finetuned_ensemble_final/slide_preds_ensemble.npy')
bulk = np.load('/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/finetuned_ensemble_final/slide_bulks.npy')
genes = np.load('/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/finetuned_ensemble_final/common_genes.npy', allow_pickle=True)
print('pred shape:', pred.shape)
print('bulk shape:', bulk.shape)
print('genes:', len(genes))
print('pred sample:', pred[0, :5])
print('bulk sample:', bulk[0, :5])

# 이전 fold_01 결과와 비교
pred_old = np.load('/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/finetuned_ensemble/slide_preds_ensemble.npy')
print('old pred sample:', pred_old[0, :5])


pred shape: (331, 2000)
bulk shape: (331, 1968)
genes: 1968
pred sample: [0.7887243  0.5061616  0.02581372 0.07704011 0.18523748]
bulk sample: [3.76210436 2.00106099 0.25619141 0.29386293 0.55106505]
old pred sample: [0.8238433  0.4620142  0.02616367 0.0831069  0.19681731]


In [2]:
# fold_10 checkpoint 확인

import torch
ckpt = torch.load(
    '/project_antwerp/hbae/Loki_output/0317_10epoch_finetune_10fold_runs_hvg_/fold_10/finetune_hvg_fold_10_20260322_021805/checkpoints/epoch_latest.pt',
    map_location='cpu', weights_only=False
)
print('epoch:', ckpt['epoch'])
print('name:', ckpt['name'])


FileNotFoundError: [Errno 2] No such file or directory: '/project_antwerp/hbae/Loki_output/0317_10epoch_finetune_10fold_runs_hvg_/fold_10/finetune_hvg_fold_10_20260322_021805/checkpoints/epoch_latest.pt'

In [4]:

import torch, glob

# 현재 사용된 것
ckpts = glob.glob('/project_antwerp/hbae/Loki_output/0317_10epoch_finetune_10fold_runs_hvg_/fold_10/finetune_hvg_*/checkpoints/epoch_latest.pt')
for c in ckpts:
    ckpt = torch.load(c, map_location='cpu', weights_only=False)
    print(f'Path: {c}')
    print(f"  epoch: {ckpt['epoch']}, name: {ckpt['name']}")


Path: /project_antwerp/hbae/Loki_output/0317_10epoch_finetune_10fold_runs_hvg_/fold_10/finetune_hvg_fold_10_20260324_183036/checkpoints/epoch_latest.pt
  epoch: 10, name: finetune_hvg_fold_10_20260324_183036


In [5]:

import numpy as np
import os

d = '/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/TCGA_embeddings/fold_10/'
files = [f for f in os.listdir(d) if not f.endswith('_coords.npy')][:3]
for f in files:
    e = np.load(d + f)
    print(f'{f}: shape={e.shape}, mean={e.mean():.4f}, std={e.std():.4f}')

# fold_01과 비교
d1 = '/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/TCGA_embeddings/fold_01/'
for f in files:
    if os.path.exists(d1 + f):
        e1 = np.load(d1 + f)
        e10 = np.load(d + f)
        print(f'{f}: fold01 mean={e1.mean():.4f} vs fold10 mean={e10.mean():.4f}, diff={abs(e1.mean()-e10.mean()):.4f}')


TCGA-WA-A7GZ-01Z-00-DX2.npy: shape=(2164, 768), mean=0.0002, std=0.0361
TCGA-CV-5978-01Z-00-DX1.npy: shape=(7575, 768), mean=0.0004, std=0.0361
TCGA-MZ-A6I9-01Z-00-DX1.npy: shape=(385, 768), mean=0.0022, std=0.0360
TCGA-WA-A7GZ-01Z-00-DX2.npy: fold01 mean=0.0006 vs fold10 mean=0.0002, diff=0.0004
TCGA-CV-5978-01Z-00-DX1.npy: fold01 mean=0.0004 vs fold10 mean=0.0004, diff=0.0000
TCGA-MZ-A6I9-01Z-00-DX1.npy: fold01 mean=0.0017 vs fold10 mean=0.0022, diff=0.0005


In [6]:

import numpy as np
import os

folds = ['fold_01', 'fold_03', 'fold_07', 'fold_10']
base = '/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/TCGA_embeddings/'
sid = 'TCGA-CV-5978-01Z-00-DX1'

embs = {}
for fold in folds:
    path = f'{base}{fold}/{sid}.npy'
    if os.path.exists(path):
        embs[fold] = np.load(path)
        print(f'{fold}: mean={embs[fold].mean():.6f}, std={embs[fold].std():.6f}')

print()
for i, f1 in enumerate(folds):
    for f2 in folds[i+1:]:
        if f1 in embs and f2 in embs:
            diff = np.abs(embs[f1] - embs[f2]).max()
            print(f'{f1} vs {f2}: max_diff={diff:.6f}')


fold_01: mean=0.000405, std=0.036082
fold_03: mean=0.000569, std=0.036080
fold_07: mean=0.000566, std=0.036080
fold_10: mean=0.000355, std=0.036083

fold_01 vs fold_03: max_diff=0.133059
fold_01 vs fold_07: max_diff=0.143891
fold_01 vs fold_10: max_diff=0.181900
fold_03 vs fold_07: max_diff=0.154736
fold_03 vs fold_10: max_diff=0.169093
fold_07 vs fold_10: max_diff=0.185856


In [7]:

import numpy as np

base = '/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/'

for fold in ['fold_01', 'fold_03', 'fold_07', 'fold_10']:
    e = np.load(f'{base}{fold}/train_img_embs.npy')
    x = np.load(f'{base}{fold}/train_exprs.npy')
    print(f'{fold}: train_embs mean={e.mean():.6f} std={e.std():.6f} shape={e.shape}')
    print(f'        train_exprs mean={x.mean():.6f} max={x.max():.4f} shape={x.shape}')


fold_01: train_embs mean=0.000349 std=0.036083 shape=(64157, 768)
        train_exprs mean=0.290607 max=8.2888 shape=(64157, 2000)
fold_03: train_embs mean=0.000501 std=0.036081 shape=(64172, 768)
        train_exprs mean=0.285707 max=8.2888 shape=(64172, 2000)
fold_07: train_embs mean=0.000250 std=0.036084 shape=(64157, 768)
        train_exprs mean=0.288054 max=8.2888 shape=(64157, 2000)
fold_10: train_embs mean=0.000259 std=0.036083 shape=(64154, 768)
        train_exprs mean=0.283519 max=8.2888 shape=(64154, 2000)


In [1]:

import numpy as np

base = '/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/'

for fold in ['fold_01', 'fold_03', 'fold_10']:
    val_e = np.load(f'{base}{fold}/val_img_embs.npy')
    val_x = np.load(f'{base}{fold}/val_exprs.npy')
    train_e = np.load(f'{base}{fold}/train_img_embs.npy')
    print(f'{fold}:')
    print(f'  val_img_embs:  {val_e.shape}  mean={val_e.mean():.6f}')
    print(f'  val_exprs:     {val_x.shape}  mean={val_x.mean():.6f}')
    print(f'  train_img_embs:{train_e.shape} mean={train_e.mean():.6f}')

    # val embedding이 train과 얼마나 다른지
    import torch, torch.nn.functional as F
    te = F.normalize(torch.tensor(train_e[:100], dtype=torch.float32), dim=-1)
    ve = F.normalize(torch.tensor(val_e[:10], dtype=torch.float32), dim=-1)
    sim = ve @ te.T
    print(f'  val-train sim (sample): mean={sim.mean():.4f} max={sim.max():.4f}')
    print()


fold_01:
  val_img_embs:  (7215, 768)  mean=0.000566
  val_exprs:     (7215, 2000)  mean=0.253819
  train_img_embs:(64157, 768) mean=0.000349
  val-train sim (sample): mean=0.1911 max=0.4150

fold_03:
  val_img_embs:  (7200, 768)  mean=0.000851
  val_exprs:     (7200, 2000)  mean=0.297410
  train_img_embs:(64172, 768) mean=0.000501
  val-train sim (sample): mean=0.5460 max=0.9312

fold_10:
  val_img_embs:  (7218, 768)  mean=-0.000087
  val_exprs:     (7218, 2000)  mean=0.316832
  train_img_embs:(64154, 768) mean=0.000259
  val-train sim (sample): mean=0.5603 max=0.9258



In [2]:

import numpy as np, torch, torch.nn.functional as F

base = '/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/'

# 모든 fold 확인
for i in range(1, 11):
    fold = f'fold_{i:02d}'
    train_e = np.load(f'{base}{fold}/train_img_embs.npy')
    val_e   = np.load(f'{base}{fold}/val_img_embs.npy')
    te = F.normalize(torch.tensor(train_e[:200], dtype=torch.float32), dim=-1)
    ve = F.normalize(torch.tensor(val_e[:20],   dtype=torch.float32), dim=-1)
    sim = ve @ te.T
    print(f'{fold}: val-train sim mean={sim.mean():.4f} max={sim.max():.4f}')


fold_01: val-train sim mean=0.1848 max=0.4554
fold_02: val-train sim mean=0.1633 max=0.4528
fold_03: val-train sim mean=0.5680 max=0.9637
fold_04: val-train sim mean=0.2045 max=0.4329
fold_05: val-train sim mean=0.1022 max=0.4437
fold_06: val-train sim mean=0.2528 max=0.5082
fold_07: val-train sim mean=0.3131 max=0.8200
fold_08: val-train sim mean=0.1185 max=0.4191
fold_09: val-train sim mean=0.1405 max=0.4455
fold_10: val-train sim mean=0.5567 max=0.9278


In [1]:

import numpy as np, torch, torch.nn.functional as F, pandas as pd
from scipy.stats import pearsonr

device = 'cuda'
GENE_LIST = '/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt'
REF_FILE  = '/project_antwerp/hbae/ref_file.csv'
FT_EMB    = '/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_01'
TCGA_EMB  = '/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/TCGA_embeddings/fold_01'

with open(GENE_LIST) as f:
    gene_list = [l.strip() for l in f if l.strip()]

ref_df = pd.read_csv(REF_FILE, index_col=0)
ref_df['slide_id'] = ref_df['wsi_file_name'].apply(lambda x: x.split('.')[0])
rna_cols     = [c for c in ref_df.columns if c.startswith('rna_')]
ref_genes    = [c.replace('rna_', '') for c in rna_cols]
common_genes = [g for g in gene_list if g in ref_genes]
common_idx   = [gene_list.index(g) for g in common_genes]
bulk_cols    = ['rna_' + g for g in common_genes]

# fold_01 train embs
train_embs = F.normalize(torch.tensor(
    np.load(f'{FT_EMB}/train_img_embs.npy'), dtype=torch.float32, device=device), dim=-1)
train_expr = torch.tensor(
    np.load(f'{FT_EMB}/train_exprs.npy'), dtype=torch.float32, device=device)

# 슬라이드 하나 선택
sid = 'TCGA-CV-6950-01Z-00-DX1'
row = ref_df[ref_df['slide_id']==sid].iloc[0]
bulk = row[bulk_cols].values.astype(float)  # (1968,)

# 모든 tile embedding 로드
embs = F.normalize(torch.tensor(
    np.load(f'{TCGA_EMB}/{sid}.npy'), dtype=torch.float32, device=device), dim=-1)
coords = np.load(f'{TCGA_EMB}/{sid}_coords.npy')

print(f'Slide: {sid}')
print(f'Tiles: {embs.shape[0]}')

# tile별 PredEx
with torch.no_grad():
    sim     = torch.clamp(embs @ train_embs.T, min=0)
    weights = sim / (sim.sum(dim=1, keepdim=True) + 1e-8)
    tile_preds = (weights @ train_expr).cpu().numpy()  # (T, 2000)

# common_genes 순서로 변환
tile_preds_common = tile_preds[:, common_idx]  # (T, 1968)

# 각 tile의 bulk와 PCC
tile_pccs = []
for i in range(len(tile_preds_common)):
    p, b = tile_preds_common[i], bulk
    if p.std() < 1e-8: 
        tile_pccs.append(np.nan)
        continue
    r, _ = pearsonr(p, b)
    tile_pccs.append(r)

tile_pccs = np.array(tile_pccs)
valid = ~np.isnan(tile_pccs)
print(f'Tile PCC: mean={tile_pccs[valid].mean():.4f} min={tile_pccs[valid].min():.4f} max={tile_pccs[valid].max():.4f}')

# top 500 tile
top500_idx = np.argsort(tile_pccs[valid])[::-1][:500]
print(f'Top 500 tile PCC: mean={tile_pccs[valid][top500_idx].mean():.4f}')
print(f'Bottom 500 tile PCC: mean={tile_pccs[valid][np.argsort(tile_pccs[valid])[:500]].mean():.4f}')
print(f'Top 500 coords sample: {coords[top500_idx[:5]]}')


Slide: TCGA-CV-6950-01Z-00-DX1
Tiles: 6237
Tile PCC: mean=0.7071 min=0.6908 max=0.7158
Top 500 tile PCC: mean=0.7113
Bottom 500 tile PCC: mean=0.7016
Top 500 coords sample: [[26624 38912]
 [52736 39936]
 [27136 38912]
 [52736 40448]
 [35328 17920]]


In [3]:

import numpy as np
import pandas as pd
from scipy.stats import pearsonr

TCGA_EMB  = '/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/TCGA_embeddings'
FT_EMB    = '/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding'
ZEROSHOT  = '/project_antwerp/hbae/Loki_output/0228_embeddings_zeroshot/fold_01'
REF_FILE  = '/project_antwerp/hbae/ref_file.csv'
GENE_LIST = '/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt'
HVG300    = '/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/tcga_bulk_hvg300_from_common.npy'
OUTPUT    = '/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/finetuned_ensemble_final_v2'

import torch, torch.nn.functional as F

with open(GENE_LIST) as f:
    gene_list = [l.strip() for l in f if l.strip()]

ref_df = pd.read_csv(REF_FILE, index_col=0)
ref_df['slide_id'] = ref_df['wsi_file_name'].apply(lambda x: x.split('.')[0])
rna_cols     = [c for c in ref_df.columns if c.startswith('rna_')]
ref_genes    = [c.replace('rna_', '') for c in rna_cols]
common_genes = [g for g in gene_list if g in ref_genes]
common_idx   = [gene_list.index(g) for g in common_genes]
bulk_cols    = ['rna_' + g for g in common_genes]

val_exprs = np.load(ZEROSHOT + '/val_exprs.npy')
top300_A  = set(gene_list[i] for i in np.argsort(val_exprs.mean(axis=0))[::-1][:300])
idx_A     = [i for i, g in enumerate(common_genes) if g in top300_A]
hvg300_B  = set(np.load(HVG300, allow_pickle=True).tolist())
idx_B     = [i for i, g in enumerate(common_genes) if g in hvg300_B]

# 저장된 결과 로드
pred_arr = np.load(OUTPUT + '/ensemble_preds.npy')  # (331, 1968)
bulk_arr = np.load(OUTPUT + '/ensemble_bulks.npy')  # (331, 1968)

def calc_gene_pcc_sum(pred, bulk, idx, label):
    p, b = pred[:, idx], bulk[:, idx]
    pccs = []
    for i in range(p.shape[1]):
        if p[:,i].std() < 1e-8 or b[:,i].std() < 1e-8:
            continue
        r, _ = pearsonr(p[:,i], b[:,i])
        pccs.append(r)
    pccs = np.array(pccs)
    print(f'[{label}]')
    print(f'  Gene PCC sum:    {pccs.sum():.4f}')
    print(f'  Gene PCC mean:   {pccs.mean():.4f}')
    print(f'  PCC > 0 genes:   {(pccs > 0).sum()} / {len(pccs)}')
    print(f'  PCC > 0.1 genes: {(pccs > 0.1).sum()}')
    return pccs

print('=== Ensemble (10 fold) ===')
calc_gene_pcc_sum(pred_arr, bulk_arr, list(range(len(common_genes))), 'All 1968')
calc_gene_pcc_sum(pred_arr, bulk_arr, idx_A, 'Method A top300')
calc_gene_pcc_sum(pred_arr, bulk_arr, idx_B, 'Method B top300')

# fold별 sum도 계산
print()
print('=== Fold-wise Gene PCC Sum (All 1968) ===')
device = torch.device('cuda')
matched = [(row['slide_id'], row[bulk_cols].values.astype(float))
           for _, row in ref_df.iterrows()
           if __import__('os').path.exists(f'{TCGA_EMB}/fold_01/{row["slide_id"]}.npy')]

import os
from collections import defaultdict
from tqdm import tqdm

fold_sums = []
for fold_num in range(1, 11):
    fold = f'fold_{fold_num:02d}'
    train_embs = F.normalize(torch.tensor(
        np.load(f'{FT_EMB}/{fold}/train_img_embs.npy'), dtype=torch.float32, device=device), dim=-1)
    train_expr = torch.tensor(
        np.load(f'{FT_EMB}/{fold}/train_exprs.npy'), dtype=torch.float32, device=device)

    preds, bulks = [], []
    for sid, bulk_vals in matched:
        emb_path = f'{TCGA_EMB}/{fold}/{sid}.npy'
        if not os.path.exists(emb_path): continue
        embs = F.normalize(torch.tensor(
            np.load(emb_path), dtype=torch.float32, device=device), dim=-1)
        with torch.no_grad():
            sim  = torch.clamp(embs @ train_embs.T, min=0)
            w    = sim / (sim.sum(dim=1, keepdim=True) + 1e-8)
            pred = (w @ train_expr).mean(dim=0).cpu().numpy()[common_idx]
        preds.append(pred)
        bulks.append(bulk_vals)

    p_arr = np.array(preds)
    b_arr = np.array(bulks)
    pccs  = [pearsonr(p_arr[:,i], b_arr[:,i])[0]
             for i in range(p_arr.shape[1])
             if p_arr[:,i].std() > 1e-8 and b_arr[:,i].std() > 1e-8]
    pccs = np.array(pccs)
    print(f'{fold}: sum={pccs.sum():.2f}  mean={pccs.mean():.4f}  PCC>0: {(pccs>0).sum()}/{len(pccs)}')
    fold_sums.append(pccs.sum())
    del train_embs, train_expr
    torch.cuda.empty_cache()

print(f'\nAvg sum across folds: {np.mean(fold_sums):.2f}')


=== Ensemble (10 fold) ===
[All 1968]
  Gene PCC sum:    72.2987
  Gene PCC mean:   0.0367
  PCC > 0 genes:   1361 / 1968
  PCC > 0.1 genes: 375
[Method A top300]
  Gene PCC sum:    13.2796
  Gene PCC mean:   0.0458
  PCC > 0 genes:   215 / 290
  PCC > 0.1 genes: 55
[Method B top300]
  Gene PCC sum:    8.6856
  Gene PCC mean:   0.0289
  PCC > 0 genes:   186 / 301
  PCC > 0.1 genes: 66

=== Fold-wise Gene PCC Sum (All 1968) ===
fold_01: sum=71.28  mean=0.0362  PCC>0: 1297/1968
fold_02: sum=88.41  mean=0.0449  PCC>0: 1421/1968
fold_03: sum=97.59  mean=0.0496  PCC>0: 1496/1968
fold_04: sum=71.98  mean=0.0366  PCC>0: 1391/1968
fold_05: sum=79.97  mean=0.0406  PCC>0: 1404/1968
fold_06: sum=79.32  mean=0.0403  PCC>0: 1412/1968
fold_07: sum=51.71  mean=0.0263  PCC>0: 1219/1968
fold_08: sum=43.65  mean=0.0222  PCC>0: 1251/1968
fold_09: sum=82.90  mean=0.0421  PCC>0: 1449/1968
fold_10: sum=-17.51  mean=-0.0089  PCC>0: 974/1968

Avg sum across folds: 64.93


In [1]:

import numpy as np
# fold_10 제외
sums = [71.28, 88.41, 97.59, 71.98, 79.97, 79.32, 51.71, 43.65, 82.90]
print(f'9-fold avg sum: {np.mean(sums):.2f}')
print(f'9-fold best (fold_03): 97.59')


9-fold avg sum: 74.09
9-fold best (fold_03): 97.59


In [1]:

import numpy as np
import torch, torch.nn.functional as F

device = 'cuda'
FT_EMB   = '/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_03'
TCGA_EMB = '/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/TCGA_embeddings/fold_03'

train_embs = F.normalize(torch.tensor(
    np.load(f'{FT_EMB}/train_img_embs.npy'), dtype=torch.float32, device=device), dim=-1)

sid    = 'TCGA-CV-6950-01Z-00-DX1'
embs   = F.normalize(torch.tensor(
    np.load(f'{TCGA_EMB}/{sid}.npy'), dtype=torch.float32, device=device), dim=-1)

# tile image embedding → ST train과 max similarity
with torch.no_grad():
    sim = embs @ train_embs.T  # (T, N_train)
    max_sim = sim.max(dim=1).values.cpu().numpy()  # 각 tile의 best match similarity
    mean_sim = sim.mean(dim=1).cpu().numpy()

print('Max sim with ST train:')
print(f'  min={max_sim.min():.4f} max={max_sim.max():.4f} mean={max_sim.mean():.4f} std={max_sim.std():.4f}')
print('Mean sim with ST train:')
print(f'  min={mean_sim.min():.4f} max={mean_sim.max():.4f} mean={mean_sim.mean():.4f} std={mean_sim.std():.4f}')


Max sim with ST train:
  min=0.4358 max=0.6768 mean=0.5645 std=0.0342
Mean sim with ST train:
  min=0.1607 max=0.3026 mean=0.2436 std=0.0180


In [1]:

import numpy as np

base = '/project_antwerp/hbae/Loki_output/0317_epoch10_finetune_embedding_new'
path = f'{base}/fold_01/Top_500_sample_gene_pcc_dist.npy'

data = np.load(path, allow_pickle=True).item()
print(type(data))

if isinstance(data, dict):
    print('Keys:', list(data.keys())[:10])
    for k, v in list(data.items())[:3]:
        print(f'  {k}: type={type(v)}, ', end='')
        if hasattr(v, 'shape'):
            print(f'shape={v.shape}')
        else:
            print(f'value={v}')


<class 'dict'>
Keys: ['GSM6339631_s1', 'GSM8633892_21_00758_LI_SING']
  GSM6339631_s1: type=<class 'numpy.ndarray'>, shape=(300,)
  GSM8633892_21_00758_LI_SING: type=<class 'numpy.ndarray'>, shape=(300,)


In [2]:

import numpy as np
import os

base = '/project_antwerp/hbae/Loki_output/0317_epoch10_finetune_embedding_new'

# gene list 파일 찾기
for root, dirs, files in os.walk(base):
    for f in files:
        if 'gene' in f.lower() and ('list' in f.lower() or 'name' in f.lower() or '.txt' in f or '.csv' in f):
            print(os.path.join(root, f))

# HVG gene list도 확인
hvg_candidates = [
    '/project_antwerp/hbae/Loki/0317_HVG_NEW/HVG_2000_genes.txt',
    '/project_antwerp/hbae/data/HVG_2000_genes.txt',
]
for p in hvg_candidates:
    if os.path.exists(p):
        print(f'Found: {p}')
